# Notebook 06 — Importancia de Atributos y Análisis de Adjetivos

## Objetivo

Interpretar qué términos lingüísticos asocian los modelos lineales del Experimento 1 con fake vs real, comparar consistencia entre LR / LinearSVC / Multinomial NB, y validar la Hipótesis 1 sobre adjetivos emocionales/sensacionalistas.

Referencia metodológica: [ADR Experimento 4](../docs/adr/experimento-04-importancia-atributos.md).

> Los coeficientes reflejan patrones del dataset, no veracidad factual.

In [1]:
import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import pandas as pd
import spacy

from nlp.interpretability import (
    adjective_sentiment_summary,
    adjective_sentiment_table,
    best_config_rows_per_model,
    extract_adjective_counts,
    feature_importance_dataframe,
    fit_bigram_interpretation_pipeline,
    load_best_baseline_pipeline,
    load_source_normalization_flag,
    prepare_text_series,
    term_overlap_summary,
    top_bigrams_table,
    top_ngrams_by_class_frequency,
    top_terms_table,
)
from nlp.io import load_splits
from nlp.modeling import (
    baseline_text_columns,
    config_from_row,
    fit_pipeline_from_config,
)
from nlp.paths import (
    RESULTS_FIGURES,
    RESULTS_METRICS,
    RESULTS_MODELS,
    SOURCE_ABLATION_DECISION,
)
from nlp.plotting import (
    plot_adjective_frequency_comparison,
    plot_top_terms_horizontal,
    plot_wordcloud_from_frequencies,
    setup_style,
)

setup_style()
FIGURE_PREFIX = "06_"
TOP_K = 15

## 1. Carga de datos, ablación de fuente y mejor baseline (politics)

Se deriva la columna de texto desde la config del mejor modelo (no se fija `full_text` a mano). Si la ablación de fuente del notebook 03 activó normalización, se aplica a los análisis descriptivos.

In [2]:
baseline_results = pd.read_csv(RESULTS_METRICS / "baseline_results.csv")
pol_train, _, pol_test = load_splits("politics", columns=baseline_text_columns())

normalize_source = load_source_normalization_flag(SOURCE_ABLATION_DECISION)
print(
    "Normalización de fuente en análisis descriptivos:",
    "sí" if normalize_source else "no",
)

model_path = RESULTS_MODELS / "best_baseline_politics.joblib"
config_path = RESULTS_MODELS / "best_baseline_politics_config.json"

pipe, best_config, TEXT_COL = load_best_baseline_pipeline(
    model_path,
    config_path,
    baseline_results,
    pol_train,
    normalize_source=normalize_source,
)

print("Mejor baseline politics:")
print(
    pd.Series(best_config)[
        ["model", "vectorizer", "text_field", "stopwords", "ngram_range", "max_features"]
    ].to_string()
)
print(f"Columna de texto para interpretación: {TEXT_COL}")


Normalización de fuente en análisis descriptivos: no
Mejor baseline politics:
model                  linear_svm
vectorizer                    bow
text_field                   body
stopwords       without_stopwords
ngram_range                (1, 2)
max_features                10000
Columna de texto para interpretación: clean_body_text_without_stopwords


## 2. Comparación multi-modelo: LR, LinearSVC y Multinomial NB

Para cada algoritmo se toma su mejor config en validación (`baseline_results.csv`), se refitea en train y se extraen los top términos. El solapamiento Jaccard entre top-K mide si los patrones son consistentes entre modelos.

> **Comparabilidad:** se fija el mismo campo de texto y vectorizador (los del mejor baseline, `body`/`bow`) para los tres modelos. Sin fijarlos, NB podía elegir `title` mientras LR/SVM usaban `body`, y el Jaccard daba 0.0 por comparar vocabularios distintos —no por inconsistencia algorítmica real—.</cell id="bac1e42e">

In [ ]:
best_per_model = best_config_rows_per_model(
    baseline_results,
    force_text_field=best_config["text_field"],
    force_vectorizer=best_config["vectorizer"],
)
importance_tables: list[pd.DataFrame] = []
terms_fake_by_model: dict[str, set[str]] = {}
terms_real_by_model: dict[str, set[str]] = {}

for _, row in best_per_model.iterrows():
    model_name = row["model"]
    config = config_from_row(row)
    model_pipe, _ = fit_pipeline_from_config(
        pol_train,
        config,
        normalize_source=normalize_source,
    )
    importance = feature_importance_dataframe(model_pipe)
    table = top_terms_table(importance, n=30, model=model_name)
    importance_tables.append(table)

    terms_fake_by_model[model_name] = set(
        importance.nlargest(TOP_K, "coefficient")["term"]
    )
    terms_real_by_model[model_name] = set(
        importance.nsmallest(TOP_K, "coefficient")["term"]
    )

feature_importance = pd.concat(importance_tables, ignore_index=True)
feature_importance.to_csv(RESULTS_METRICS / "feature_importance.csv", index=False)

overlap_fake = term_overlap_summary(terms_fake_by_model)
overlap_real = term_overlap_summary(terms_real_by_model)
overlap_fake["direction"] = "fake"
overlap_real["direction"] = "real"
term_overlap = pd.concat([overlap_fake, overlap_real], ignore_index=True)
term_overlap.to_csv(RESULTS_METRICS / "feature_importance_overlap.csv", index=False)

print("Mejor config por modelo (val), fijando campo/vectorizador del mejor baseline:")
print(
    best_per_model[
        ["model", "vectorizer", "text_field", "f2_fake", "ngram_range"]
    ].to_string(index=False)
)
print("\nSolapamiento Jaccard (top términos fake):")
print(overlap_fake.to_string(index=False))
print("\nSolapamiento Jaccard (top términos real):")
print(overlap_real.to_string(index=False))

best_importance = feature_importance_dataframe(pipe)
top_fake = best_importance.nlargest(30, "coefficient")[["term", "coefficient"]]
top_real = best_importance.nsmallest(30, "coefficient")[["term", "coefficient"]]
feature_importance.head(10)

In [4]:
plot_top_terms_horizontal(
    top_fake,
    top_real,
    n=TOP_K,
    save_path=RESULTS_FIGURES / f"{FIGURE_PREFIX}feature_importance_top_terms.png",
)
plt.show()


## 3. Bigramas predictivos vs frecuencia descriptiva

Los **bigramas con mayor peso** en el modelo (coeficientes de términos con espacio) responden al ADR. La frecuencia por clase es contexto descriptivo complementario.

In [5]:
bigram_pipe, _ = fit_bigram_interpretation_pipeline(
    pol_train,
    best_config,
    normalize_source=normalize_source,
)
bigram_importance = feature_importance_dataframe(bigram_pipe)
bigram_weights = top_bigrams_table(
    bigram_importance,
    n=20,
    model=best_config["model"],
)
bigram_weights.to_csv(RESULTS_METRICS / "bigram_importance.csv", index=False)

analysis_df = pd.concat([pol_train, pol_test], ignore_index=True)
analysis_text = prepare_text_series(
    analysis_df[TEXT_COL],
    normalize_source=normalize_source,
)
analysis_df = analysis_df.assign(_analysis_text=analysis_text)

fake_bi_freq = top_ngrams_by_class_frequency(
    analysis_df,
    "_analysis_text",
    1,
    normalize_source=False,
)
real_bi_freq = top_ngrams_by_class_frequency(
    analysis_df,
    "_analysis_text",
    0,
    normalize_source=False,
)

print("Top bigramas por PESO del modelo (fake):", bigram_weights[bigram_weights["direction"] == "fake"].head(10).to_dict("records"))
print("Top bigramas por PESO del modelo (real):", bigram_weights[bigram_weights["direction"] == "real"].head(10).to_dict("records"))
print("\nBigramas frecuentes FAKE (descriptivo):", fake_bi_freq[:10])
print("Bigramas frecuentes REAL (descriptivo):", real_bi_freq[:10])


Top bigramas por PESO del modelo (fake): [{'term': 'cia director', 'coefficient': 0.1462046195514572, 'direction': 'fake', 'model': 'linear_svm'}, {'term': 'told reuters', 'coefficient': 0.13504070260160717, 'direction': 'fake', 'model': 'linear_svm'}, {'term': 'reuters ipsos', 'coefficient': 0.124121227269644, 'direction': 'fake', 'model': 'linear_svm'}, {'term': 'twitter com', 'coefficient': 0.12050562739366497, 'direction': 'fake', 'model': 'linear_svm'}, {'term': 'breitbart news', 'coefficient': 0.1089204816740368, 'direction': 'fake', 'model': 'linear_svm'}, {'term': 'pic twitter', 'coefficient': 0.09853609964272755, 'direction': 'fake', 'model': 'linear_svm'}, {'term': 'anyone else', 'coefficient': 0.08655849472353307, 'direction': 'fake', 'model': 'linear_svm'}, {'term': 'obama administration', 'coefficient': 0.07999733218778804, 'direction': 'fake', 'model': 'linear_svm'}, {'term': 'president trump', 'coefficient': 0.07403396306613337, 'direction': 'fake', 'model': 'linear_svm'

In [6]:
top_fake_bi = bigram_weights[bigram_weights["direction"] == "fake"].head(TOP_K)
top_real_bi = bigram_weights[bigram_weights["direction"] == "real"].head(TOP_K)

plot_top_terms_horizontal(
    top_fake_bi,
    top_real_bi,
    n=TOP_K,
    save_path=RESULTS_FIGURES / f"{FIGURE_PREFIX}bigram_importance_top_terms.png",
)
plt.show()

## 4. Adjetivos con spaCy y carga semántica (valencia léxica VADER)

Validación de la Hipótesis 1: adjetivos con mayor carga emocional en fake. Se usa la misma columna que el modelo (`body`) y train+test para la lectura cualitativa (vocabulario aprendido en train; ejemplos de test incluidos).

> La "carga semántica" se mide con la **valencia por palabra del lexicon de VADER** (`vader.lexicon`, rango ~−4..+4), no con `polarity_scores()` sobre el adjetivo aislado: VADER es un analizador a nivel oración cuyo *compound* depende de mayúsculas, signos y modificadores —ausentes en una palabra suelta—, por lo que devolvía 0.0 para la mayoría de los lemas. Se reporta además `n_con_valencia` (cuántos de los top adjetivos tienen entrada con valencia ≠ 0 en el lexicon).

In [ ]:
nlp = spacy.load("en_core_web_sm")

adj_analysis_df = analysis_df[[ "label", "_analysis_text"]].copy()
fake_texts = adj_analysis_df.loc[adj_analysis_df["label"] == 1, "_analysis_text"]
real_texts = adj_analysis_df.loc[adj_analysis_df["label"] == 0, "_analysis_text"]

fake_adj = extract_adjective_counts(nlp, fake_texts, sample_size=3000)
real_adj = extract_adjective_counts(nlp, real_texts, sample_size=3000)

fake_adj_df = adjective_sentiment_table(fake_adj, class_label="fake")
real_adj_df = adjective_sentiment_table(real_adj, class_label="real")
adj_df = pd.concat([fake_adj_df, real_adj_df], ignore_index=True)
adj_df.to_csv(RESULTS_METRICS / "adjectives_by_class.csv", index=False)

adj_summary = adjective_sentiment_summary(adj_df)
adj_summary.to_csv(RESULTS_METRICS / "adjectives_sentiment_summary.csv", index=False)

print("Carga emocional por clase (valencia léxica VADER, ponderada por frecuencia):")
print(adj_summary.to_string(index=False))
adj_df.head(10)

In [ ]:
# Hipótesis 1: carga emocional de adjetivos por clase (|valencia| léxica VADER,
# ponderada por frecuencia). Visualiza que fake ≈ 2× la valencia absoluta de real.
fig, ax = plt.subplots(figsize=(6, 4))
order = adj_summary.set_index("class").reindex(["fake", "real"])
colors = {"fake": "#e74c3c", "real": "#2ecc71"}
bars = ax.bar(
    order.index,
    order["mean_valence_abs_weighted"],
    color=[colors[c] for c in order.index],
)
for bar, value in zip(bars, order["mean_valence_abs_weighted"]):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.005, f"{value:.3f}", ha="center")
ax.set_ylabel("|valencia| léxica VADER (ponderada por frecuencia)")
ax.set_title("Hipótesis 1: carga emocional de adjetivos (fake ≈ 2× real)")
save_figure(fig, RESULTS_FIGURES / f"{FIGURE_PREFIX}h1_adjective_valence.png")
plt.show()

In [8]:
plot_adjective_frequency_comparison(
    fake_adj,
    real_adj,
    top_n=TOP_K,
    save_path=RESULTS_FIGURES / f"{FIGURE_PREFIX}adjectives_by_class.png",
)
plt.show()

for class_label, counter, colormap in [
    ("fake", fake_adj, "Reds"),
    ("real", real_adj, "Greens"),
]:
    fig = plot_wordcloud_from_frequencies(
        counter,
        title=f"Adjetivos — {class_label}",
        colormap=colormap,
        save_path=RESULTS_FIGURES / f"{FIGURE_PREFIX}adjectives_wordcloud_{class_label}.png",
    )
    if fig is not None:
        plt.show()
    else:
        print("wordcloud no instalado; se omite word cloud de adjetivos")


## 5. Conclusiones

- **Términos y bigramas predictivos:** el mejor baseline (LinearSVC, BoW, `body`) asocia con **fake** expresiones de redes sociales y *calls to action* (`via`, `read`, `watch`, `pic twitter`, `twitter com`, `breitbart news`, `daily mail`) y con **real**, mayormente **marcadores de fuente** (`washington reuters`, `moscow reuters`, `reuters president`, `york reuters`).

- **Cuidado — sesgo de fuente sin normalizar:** la ablación del Notebook 03 dejó `use_source_normalization` en el valor de `source_ablation_decision.json` (en la corrida actual, `false`: la caída de F2 al normalizar quedó por debajo del umbral de 0,03). Por eso estos coeficientes se leen sobre texto **sin normalizar**, y varios términos top "real" son marcas de agencia (`reuters`), no estilo periodístico lingüístico: la señal de fuente —aunque débil para clasificar— domina los coeficientes negativos. No deben presentarse como "estilo periodístico" puro. (Si una corrida con la lista de marcadores ampliada activara la normalización, este análisis se reharía sobre texto con `[SOURCE]`.)

- **Consistencia multi-modelo (mismo campo/vectorizador):** fijando `body`/`bow` para los tres modelos, **LR y LinearSVC son muy consistentes** (Jaccard 0.50 fake / 0.25 real). **Multinomial NB diverge** (Jaccard ≈ 0.0 fake), pero ya **no** por comparar campos distintos —están fijados—, sino porque NB rankea por *log-odds* (favorece términos de alta frecuencia condicional) mientras LR/SVM rankean por margen discriminativo. Es una diferencia algorítmica genuina, no un artefacto de la comparación.

- **Hipótesis 1 (adjetivos) — SOSTENIDA en dirección.** Medida con la **valencia léxica de VADER por palabra** (ponderada por frecuencia), la carga emocional absoluta de los adjetivos fake (|valencia| = **0.296**) **duplica** la de los reales (**0.118**). Los adjetivos fake de mayor valencia son marcadamente evaluativos (`great` +3.1, `free` +2.3, `illegal` −2.6, `bad` −2.5, `criminal` −2.4); los reales, más neutros (`legal`, `low`, `clear`). **Matiz:** la mayoría de los adjetivos más frecuentes de ambas clases son neutros (solo ~7/50 fake y ~6/50 real tienen valencia ≠ 0 en el lexicon), de modo que la diferencia la marcan unos pocos adjetivos cargados que aparecen más en fake. La evidencia respalda H1, sin sobre-afirmarla.

- **Limitación:** estas asociaciones son correlaciones estilísticas del corpus político (la clase real proviene de una única fuente, Reuters); no implican verificación factual ni aplican a DistilBERT (caja negra → Experimento 5). Ver la sección de limitaciones del Informe.